# Part 2 · Indexing and evaluation

This notebook goes with the Part 2 report (`reports/part-2/main.tex`), and its
sections follow the report's. Reusable code lives in the package:
`fashion_search.index` builds the inverted index, `fashion_search.ranking`
ranks the matches by TF-IDF and `fashion_search.evaluation` holds the metrics.
This notebook runs them and saves the report's figures to `reports/part-2/figures/`.

Run `uv run python scripts/fetch_data.py` and `uv run python -m fashion_search.corpus`
first, then run the notebook top to bottom.

In [70]:
import sys
import os

# Go up one level from 'notebooks' to the root directory, then into 'src'
src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from fashion_search import corpus as cp
from fashion_search import evaluation as ev
from fashion_search.index import match
from fashion_search.ranking import SearchEngine
from fashion_search.text import build_terms

FIGURES = Path("../reports/part-2/figures")
FIGURES.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 80)

raw = cp.load_raw()
products = cp.load_corpus()
engine = SearchEngine(products)
by_pid = products.set_index("pid")
VALIDATION_QUERIES = {1: "women full sleeve sweatshirt cotton", 2: "men slim jeans blue"}
validation_labels = ev.load_labels("../data/validation_labels.csv")
len(products)

28080

## 2. Indexing

### 2.1 Inverted index

**Brand names completed.** Part 1 found that most brand names are cut off in
the source (`Pu`, `True Bl`), so a search for *puma* found nothing.
`corpus.complete_brand` completes them from the product URL, which starts with
the full brand (`/puma-solid-men-...`). Sellers also lose the rating and
notice glued to some of them (`RetailNet4.5Seller changed...`).

In [71]:
changed = raw.brand.notna() & (raw.brand != products.brand)
print(f"products with a completed brand: {changed.sum():,} of {len(products):,}")
(pd.DataFrame({"source": raw.brand[changed], "completed": products.brand[changed]})
   .value_counts().head(10).rename("products").to_frame())

products with a completed brand: 26,798 of 28,080


,,products
source,completed,
ECKO Unl,ECKO Unltd,951
Free Authori,Free Authority,860
ARBO,ARBOUR,806
REEB,REEBOK,802
Pu,Puma,798
True Bl,True Blue,793
Keo,Keoti,660
Amp,Ample,585
Black Beat,Black Beatle,548


The index maps each term to the products that contain it, with how often
(`{term: {pid: count}}`), over the four searchable fields of Part 1 (title,
description, product details and brand) merged into one text per product.

In [72]:
postings = pd.Series({term: len(posting) for term, posting in engine.index.items()})
pd.Series({
    "products": len(products),
    "terms": len(postings),
    "postings (term, product pairs)": int(postings.sum()),
    "median products per term": postings.median(),
    "terms in a single product": int((postings == 1).sum()),
}).to_frame("value")

,value
products,28080.0
terms,6884.0
"postings (term, product pairs)",905629.0
median products per term,4.0
terms in a single product,2043.0


In [73]:
postings.sort_values(ascending=False).head(10).rename("products").to_frame()

,products
wear,20800
wash,20445
cotton,19889
western,19104
sleev,18031
machin,15710
regular,15372
neck,14079
women,13807
blend,13572


Conjunctive (AND) matching keeps only the products that contain every query
term. Each term narrows the set; intersecting from the shortest posting list
first keeps every step small.

In [74]:
rows = []
for qid, query in VALIDATION_QUERIES.items():
    terms = build_terms(query)
    for i, term in enumerate(terms, start=1):
        rows.append({"query": qid, "term": term, "products with term": len(engine.index.get(term, {})),
                     "products with all terms so far": len(match(engine.index, terms[:i]))})
pd.DataFrame(rows).set_index(["query", "term"])

products with term  products with all terms so far
query term                                                          
1     women                    13807                           13807
      full                      7171                            3537
      sleev                    18031                            3479
      sweatshirt                1419                             671
      cotton                   19889                             498
2     men                      13517                           13517
      slim                      5654                            2861
      jean                      2436                             368
      blue                      7696                             225

### 2.2 TF-IDF ranking

Each term of a product or query is weighted $(1 + \log tf) \cdot \log(N / df)$,
and a product's score is the cosine similarity between its vector and the
query's. The top results of the validation queries:

In [75]:
for qid, query in VALIDATION_QUERIES.items():
    print(f"Q{qid}: {query} ({len(engine.search(query))} matches)")
    display(engine.results(query, k=10))

Q1: women full sleeve sweatshirt cotton (498 matches)


,pid,title,brand,sub_category,score
0,SWSFZVTTQCB4SJ7F,Full Sleeve Solid Women Sweatshirt,Levis,Winter Wear,0.730214
1,SWSFZ2H4KMYXZXX7,Full Sleeve Solid Women Sweatshirt,ECKO Unltd,Winter Wear,0.717517
2,SWSFY38ADYPVZHYZ,Full Sleeve Printed Women Sweatshirt,ECKO Unltd,Winter Wear,0.713111
3,SWSFZVTNGM4HG8BC,Full Sleeve Printed Women Sweatshirt,Levis,Winter Wear,0.711368
4,SWSFYRKYAHH4HHSM,Full Sleeve Printed Women Sweatshirt,Steenbok,Winter Wear,0.702167
5,SWSFYRH2F7NHW63A,Full Sleeve Solid Women Sweatshirt,Steenbok,Winter Wear,0.687806
6,SWSFZ2H3CXFAWYPH,Full Sleeve Solid Women Sweatshirt,ECKO Unltd,Winter Wear,0.673529
7,SWSFXV6FKYFF35EW,Full Sleeve Printed Women Sweatshirt,ECKO Unltd,Winter Wear,0.671752
8,SWSFYTYMNTBNARUN,Full Sleeve Solid Women Sweatshirt,ARBOUR,Winter Wear,0.670498
9,SWSFQGS456JAZCQB,Full Sleeve Printed Women Sweatshirt,Levis,Winter Wear,0.669152


Q2: men slim jeans blue (225 matches)


,pid,title,brand,sub_category,score
0,JEAFSKYHZHSZZC9S,Slim Men Blue Jeans,ECKO Unltd,Bottomwear,0.547351
1,JEAFEKQZ4C2Z6GCX,Slim Men Blue Jeans,True Blue,Bottomwear,0.536252
2,JEAFEKQZQPYHHVGS,Slim Men Blue Jeans,True Blue,Bottomwear,0.536252
3,JEAFQF6JBUSEXHVF,Slim Men Blue Jeans,ECKO Unltd,Bottomwear,0.520203
4,JEAFSKYHSYMTKEJD,Slim Men Blue Jeans,ECKO Unltd,Bottomwear,0.518406
5,JEAFM3Y9ZNMVG6ZM,Slim Men Blue Jeans,True Blue,Bottomwear,0.509574
6,JEAFEKQZNAWHW8PT,Slim Men Dark Blue Jeans,True Blue,Bottomwear,0.495771
7,JEAFRAQXEKGUPNUN,Slim Men Blue Jeans,Levis,Bottomwear,0.490248
8,JEAFEKQZWYPVGYCJ,Slim Men Blue Jeans,True Blue,Bottomwear,0.477636
9,JEAFESND4QWQUBZD,Slim Men Blue Jeans,True Blue,Bottomwear,0.470031


**Runs to evaluate.** `validation_labels.csv` judges only 20 products per
query, and none of them is in the top 10 above, so counting unjudged products
as non-relevant would give every metric a score of zero. We evaluate the
ranking of the judged products instead: each run is the judged products the
engine returns, in the engine's order. Judged products it does not return (they
lack a query term) count as relevant but not retrieved.

In [76]:
validation_runs = {
    qid: ([pid for pid in engine.search(query) if pid in validation_labels[qid]], validation_labels[qid])
    for qid, query in VALIDATION_QUERIES.items()
}
pd.DataFrame([
    {"query": qid, "judged": len(labels), "relevant": sum(labels.values()),
     "judged retrieved": len(ranked), "relevant retrieved": sum(labels[p] for p in ranked),
     "labels in ranked order": "".join(str(labels[p]) for p in ranked)}
    for qid, (ranked, labels) in validation_runs.items()
]).set_index("query")

,judged,relevant,judged retrieved,relevant retrieved,labels in ranked order
query,,,,,
1,20,13,20,13,11111111111100000010
2,20,10,8,6,11110110


### 2.3 Test queries

We build the five test queries from mid-IDF terms: terms in 300 to 6,000
products are specific enough to tell products apart, yet common enough that a
query using them returns results. Each query targets a different kind of garment.

In [77]:
term_stats = pd.DataFrame({
    "df": postings,
    "idf": pd.Series(engine.idf),
    "total_tf": pd.Series({term: sum(posting.values()) for term, posting in engine.index.items()}),
})
print("Term statistics (top 20 by document frequency):")
display(term_stats.sort_index().sort_values("df", ascending=False, kind="stable").head(20).round(3))

mid_idf = term_stats[term_stats.df.between(300, 6000)]
mid_idf.sort_index().sort_values("idf", ascending=False, kind="stable").head(30).round(3)

Term statistics (top 20 by document frequency):


,df,idf,total_tf
wear,20800,0.300,29342
wash,20445,0.317,26452
cotton,19889,0.345,33958
western,19104,0.385,19175
sleev,18031,0.443,25734
machin,15710,0.581,17908
regular,15372,0.603,27345
neck,14079,0.690,40466
women,13807,0.710,28043
blend,13572,0.727,15269


,df,idf,total_tf
way,300,4.539,323
construct,302,4.532,310
fleximaa,302,4.532,616
moistur,307,4.516,338
outing,308,4.513,405
introduc,310,4.506,310
mild,310,4.506,314
right,310,4.506,337
ankl,313,4.497,699
lay,313,4.497,313


In [78]:
TEST_QUERIES = {
    3: "women printed kurta cotton",          # ethnic wear
    4: "men polo tshirt regular fit",         # topwear, many matches
    5: "slim fit jogger men",                 # bottomwear
    6: "puma men tshirt",                     # topwear with brand
    7: "women denim jacket",                  # jackets
}
pd.DataFrame([
    {"query": qid, "text": query, "matches": len(engine.search(query)),
     "terms (IDF)": ", ".join(f"{t} ({engine.idf.get(t, 0):.2f})" for t in build_terms(query))}
    for qid, query in TEST_QUERIES.items()
]).set_index("query")

,text,matches,terms (IDF)
query,,,
3,women printed kurta cotton,40,"women (0.71), print (1.05), kurta (3.58), cotton (0.34)"
4,men polo tshirt regular fit,673,"men (0.73), polo (2.22), tshirt (0.77), regular (0.60), fit (0.95)"
5,slim fit jogger men,33,"slim (1.60), fit (0.95), jogger (3.92), men (0.73)"
6,puma men tshirt,165,"puma (3.56), men (0.73), tshirt (0.77)"
7,women denim jacket,82,"women (0.71), denim (2.76), jacket (3.23)"


Our first fifth query was *women ankle leggings*. No product title contains
*legging*, and *leggings* stems to *leg*, so it returns track pants and jeans
that mention the leg in their details. With nothing relevant to find, every
metric is 0 whatever the ranking, so we replaced it with *women denim jacket*
and discuss the case among the limitations.

In [79]:
print("titles containing 'legging':", products.title.str.contains("legging", case=False).sum())
print("terms:", build_terms("women ankle leggings"))
engine.results("women ankle leggings", k=5)

titles containing 'legging': 0
terms: ['women', 'ankl', 'leg']


,pid,title,brand,sub_category,score
0,TKPFK3W6NVCGUTHR,Solid Women Black Track Pants,REEBOK CLASSICS,Bottomwear,0.292564
1,TKPFK3W62HWUHJKK,Solid Women Dark Blue Track Pants,REEBOK CLASSICS,Bottomwear,0.289101
2,JEAFE577DMBVDVYU,Slim Women Multicolor Jeans (Pack of 3),NEBRASKA,Bottomwear,0.242287
3,JEAFE577X4HYUHF3,Slim Women Multicolor Jeans (Pack of 3),NEBRASKA,Bottomwear,0.242287
4,JEAET8SSCA3PEGGC,Stretchable Slim Women Dark Blue Jeans,NEBRASKA,Bottomwear,0.240782


**Relevance labels.** We judge the top 40 results of each test query (query 5 has only 33 matches). A title
rule writes a first draft of `data/test_labels.csv` (words are matched whole,
so *women* does not count as *men*); we then read every title and corrected
its label by hand. Only results that are not judged yet get a draft
label, unless `OVERWRITE_LABELS` is set, so re-running the notebook never
erases the hand judgements. We first judged the top 20 and later extended the
judgements to the top 40, so that relevant products ranked below 20 also count
in recall.

In [80]:
import re

LABELS_PATH = Path("../data/test_labels.csv")
OVERWRITE_LABELS = False  # True rewrites the file from the rule and loses the hand judgements


def title_words(title):
    """The lowercase words of a title, so 'women' is not matched as 'men'."""
    return set(re.findall(r"[a-z]+", title.lower()))


DRAFT_RULES = {
    3: lambda w: "kurta" in w and "women" in w,
    4: lambda w: "polo" in w and "men" in w,
    5: lambda w: ("jogger" in w or "track" in w) and "men" in w,
    6: lambda w: "shirt" in w and "men" in w,
    7: lambda w: "jacket" in w and "denim" in w and "women" in w,
}

JUDGE_DEPTH = 40

judged = pd.DataFrame(columns=["title", "pid", "query_id", "labels"])
if LABELS_PATH.exists() and not OVERWRITE_LABELS:
    judged = pd.read_csv(LABELS_PATH)
rows = []
for qid, query in TEST_QUERIES.items():
    done = set(judged.pid[judged.query_id == qid])
    for _, result in engine.results(query, k=JUDGE_DEPTH).iterrows():
        if result.pid not in done:
            rows.append({"title": result.title, "pid": result.pid, "query_id": qid,
                         "labels": int(DRAFT_RULES[qid](title_words(result.title)))})
if rows:
    pd.concat([judged, pd.DataFrame(rows)]).sort_values("query_id", kind="stable").to_csv(LABELS_PATH, index=False)
    print(f"Added {len(rows)} draft labels to {LABELS_PATH.name}: judge them by hand before evaluating.")
else:
    print(f"Every top-{JUDGE_DEPTH} result is judged in {LABELS_PATH.name}.")

pd.read_csv(LABELS_PATH).groupby("query_id").labels.agg(relevant="sum", judged="count")

Every top-40 result is judged in test_labels.csv.


,relevant,judged
query_id,,
3,40,40
4,38,40
5,10,33
6,40,40
7,34,40


## 3. Evaluation

### 3.1 Validation queries

Numbers only, rounded to three decimals.

In [81]:
K_VALUES = [5, 10, 20]
display(ev.evaluate(validation_runs, K_VALUES))
print(f"MAP: {ev.mean_average_precision(validation_runs.values()):.3f}")
print(f"MRR: {ev.mean_reciprocal_rank(validation_runs.values()):.3f}")

P@K    R@K   AP@K   F1@K  NDCG@K
query K                                    
1     5   1.00  0.385  1.000  0.556   1.000
      10  1.00  0.769  1.000  0.870   1.000
      20  0.65  1.000  0.976  0.788   0.994
2     5   0.80  0.400  0.800  0.533   0.869
      10  0.60  0.600  0.569  0.600   0.716
      20  0.30  0.600  0.569  0.400   0.716

MAP: 0.772
MRR: 1.000


### 3.2 Test queries

In [82]:
test_labels = ev.load_labels(LABELS_PATH)
test_runs = {
    qid: ([pid for pid in engine.search(query) if pid in test_labels[qid]], test_labels[qid])
    for qid, query in TEST_QUERIES.items()
}
display(ev.evaluate(test_runs, K_VALUES))
print(f"MAP: {ev.mean_average_precision(test_runs.values()):.3f}")
print(f"MRR: {ev.mean_reciprocal_rank(test_runs.values()):.3f}")

P@K    R@K   AP@K   F1@K  NDCG@K
query K                                   
3     5   1.0  0.125  1.000  0.222   1.000
      10  1.0  0.250  1.000  0.400   1.000
      20  1.0  0.500  1.000  0.667   1.000
4     5   0.8  0.105  0.710  0.186   0.830
      10  0.9  0.237  0.790  0.375   0.890
      20  0.9  0.474  0.787  0.621   0.889
5     5   1.0  0.500  1.000  0.667   1.000
      10  0.8  0.800  0.758  0.800   0.857
      20  0.5  1.000  0.923  0.667   0.978
6     5   1.0  0.125  1.000  0.222   1.000
      10  1.0  0.250  1.000  0.400   1.000
      20  1.0  0.500  1.000  0.667   1.000
7     5   1.0  0.147  1.000  0.256   1.000
      10  1.0  0.294  1.000  0.455   1.000
      20  1.0  0.588  1.000  0.741   1.000

MAP: 0.964
MRR: 1.000


## 4. Limitations

**AND matching misses relevant products that phrase a query word differently,
or not at all.** The relevant products of query 2 that it does not return:

In [83]:
query = VALIDATION_QUERIES[2]
returned = set(engine.search(query))
missed = [p for p, label in validation_labels[2].items() if label and p not in returned]
pd.DataFrame([
    {"title": by_pid.loc[p, "title"],
     "missing terms": [t for t in build_terms(query) if t not in cp.all_terms(by_pid.loc[p])]}
    for p in missed
])

,title,missing terms
0,Slim Women Blue Jeans,[men]
1,Super Skinny Women Blue Jeans,[men]
2,Tapered Fit Men Blue Jeans,[slim]
3,Super Skinny Women Blue Jeans,[men]


**Synonyms split the catalogue.** Products use different words for the same
thing, or write the same word apart or joined, and a query only finds the
products that use its form:

In [84]:
pairs = [("sweater", "sweatshirt"), ("jogger", "trackpant"), ("sneakers", "shoes"), ("track pants", "trackpants")]
pd.DataFrame([{"word": a, "matches": len(engine.search(a)), "other word": b,
               "other word matches": len(engine.search(b)),
               "both": len(engine.search(f"{a} {b}"))} for a, b in pairs])

,word,matches,other word,other word matches,both
0,sweater,149,sweatshirt,1419,10
1,jogger,559,trackpant,358,42
2,sneakers,780,shoes,526,49
3,track pants,1707,trackpants,358,352


**Near-identical products fill the top results.** Many products share a
title and differ only in colour or seller, so the top 20 shows few distinct
items; nothing in TF-IDF rewards variety, rating or availability:

In [85]:
rows = []
for qid, query in VALIDATION_QUERIES.items():
    top = by_pid.loc[engine.search(query, k=20)]
    rows.append({"query": qid, "distinct titles in top 20": top.title.nunique(),
                 "out of stock in top 20": int(top.out_of_stock.sum()),
                 "mean rating in top 20": round(top.average_rating.mean(), 2),
                 "mean rating of all matches": round(by_pid.loc[engine.search(query)].average_rating.mean(), 2)})
pd.DataFrame(rows).set_index("query")

,distinct titles in top 20,out of stock in top 20,mean rating in top 20,mean rating of all matches
query,,,,
1,3,4,3.54,3.81
2,4,2,3.42,3.29


**Common words carry little weight but still decide matching.** Terms such as
*wear* or *cotton* appear in most products, so their IDF is close to zero,
yet a query with them still excludes every product that lacks them:

In [86]:
idf = pd.Series(engine.idf).sort_values()
idf.head(10).round(3).rename("idf").to_frame()

,idf
wear,0.300
wash,0.317
cotton,0.345
western,0.385
sleev,0.443
machin,0.581
regular,0.603
neck,0.690
women,0.710
blend,0.727
